In [ ]:
!pip install datasets


In [ ]:
!pip install llama-cpp-python --upgrade --force-reinstall --prefer-binary
!pip install huggingface_hub


In [ ]:
!pip install transformers torch


In [1]:
from datasets import load_dataset

ds = load_dataset("NLP-FBK/dyspnea-crf-train")

c:\Users\kocak\Desktop\CFR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
df = ds["en"].to_pandas()
df.head()

,document_id,clinical_note,annotations
0,1017490_en,"TRIAGE: \nReports having fallen accidentally, ...","[{'ground_truth': 'unknown', 'item': 'chronic ..."
1,1584581_en,"75-year-old patient, hypertensive on therapy, ...","[{'ground_truth': 'unknown', 'item': 'chronic ..."
2,1614319_en,Patient is a nursing home resident.\nPatient a...,"[{'ground_truth': 'unknown', 'item': 'chronic ..."
3,1823273_en,Admitted to the Emergency Department for pain ...,"[{'ground_truth': 'unknown', 'item': 'chronic ..."
4,661135_en,"72-year-old male, family history, former smoke...","[{'ground_truth': 'unknown', 'item': 'chronic ..."


In [3]:
features = [pair["item"] for pair in df["annotations"][0]]
print(features)
print(f"total features: {len(features)}")

for i in range(df.shape[0]):
  annotated_count = [pair["ground_truth"] != 'unknown'  for pair in df["annotations"][i]]
  print(sum(annotated_count))

['chronic pulmonary disease', 'chronic respiratory failure', 'chronic cardiac failure', 'chronic renal failure', 'chronic metabolic failure', 'chronic rheumatologic disease', 'active neoplasia', 'chronic dialysis', "duration of the patient's consciousness recovery", "duration of the patient's unconsciousness", 'first episod of epilepsy', 'known history of epilepsy', 'history of allergy', 'history of recent trauma', 'pregnancy', 'history of drug abuse', 'history of alcohol abuse', 'anticoagulants or antiplatelet drug therapy', 'presence of prodromal symptoms', 'compliance with antiepileptic therapy', 'tloc during effort', 'tloc while supine', 'antiepileptic therapy already in place', 'drowsiness, confusion, disorientation as postcritical state', 'stiffness during the episode', 'drooling during the episode', 'tonic-clonic seizures', 'poly-pharmacological therapy', 'pale skin during the episode', 'eye deviation during the episode', 'diffuse vascular disease', 'neuropsychiatric disorders',

In [4]:
report1 = df["clinical_note"][0]
print(report1)

labels = df["annotations"][0]
print(labels)

TRIAGE: 
Reports having fallen accidentally, impacting:
- occipital region with no loss of consciousness or concussion
- right trochanter 
- right knee
- right ankle
CS 15_Cincinnati negative_fluent speech_isochoric, isocyclic, and photoreactive pupils. No rigidity.
Conscious, lucid, oriented, not agitated.
wearing rigid cervical collar
BP 120/80 mm/hg, SpO2 99% on room air, RF 14 apm, HR 80 bpm r

MEDICAL ASSESSMENT: 
reports accidental fall to the ground at home following loss of support on right leg (spontaneous femur fracture?). Impact to the right side of the body, as described in triage. Currently experiencing pain in the right femur and right elbow. Head trauma without loss of consciousness. No vomiting after the episode. 
No other details at the moment

Past Medical History: 
- Arterial hypertension
- CAD (reports quadruple BPAC)
- Diabetes mellitus type 2
- Left hip prosthesis

Therapy: metformin 500 mg, iperten 1/2 tablet, enalapril 5 mg, omnic 0.4 mg, seloken 100 mg 1/2 tabl

# STEP 1: Extract Medical Entities

In [5]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import os
os.environ["TRANSFORMERS_NO_CHAT_TEMPLATES"] = "1"

# Model name
MODEL_NAME = "blaze999/Medical-NER"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code = False)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME,trust_remote_code = False)

# Create NER pipeline
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"  # groups subword tokens into full entities
)


Device set to use cpu


In [6]:
# Example medical note
note = report1

# Run NER
entities = ner_pipeline(note)

# Pretty print results
for ent in entities:
    print({
        "text": ent["word"],
        "label": ent["entity_group"],
        "confidence": round(ent["score"], 3),
        "start": ent["start"],
        "end": ent["end"]
    })

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'text': 'fallen', 'label': 'THERAPEUTIC_PROCEDURE', 'confidence': 0.107, 'start': 23, 'end': 30}
{'text': 'occipital region', 'label': 'BIOLOGICAL_STRUCTURE', 'confidence': 0.964, 'start': 57, 'end': 74}
{'text': 'loss of consciousness', 'label': 'SIGN_SYMPTOM', 'confidence': 0.947, 'start': 82, 'end': 104}
{'text': 'concussion', 'label': 'DISEASE_DISORDER', 'confidence': 0.774, 'start': 107, 'end': 118}
{'text': 'right trochanter', 'label': 'BIOLOGICAL_STRUCTURE', 'confidence': 0.974, 'start': 120, 'end': 137}
{'text': 'right knee', 'label': 'BIOLOGICAL_STRUCTURE', 'confidence': 0.93, 'start': 140, 'end': 151}
{'text': 'right ankle', 'label': 'BIOLOGICAL_STRUCTURE', 'confidence': 0.923, 'start': 153, 'end': 165}
{'text': 'CS 15_Cincinnati', 'label': 'DETAILED_DESCRIPTION', 'confidence': 0.805, 'start': 165, 'end': 182}
{'text': 'negative_fluent speech_', 'label': 'DETAILED_DESCRIPTION', 'confidence': 0.572, 'start': 182, 'end': 206}
{'text': 'isochoric', 'label': 'SIGN_SYMPTOM', 'con

In [7]:
def load_medical_model():
    """
    Downloads and loads the Medical Qwen3 model for medical entity extraction.
    
    Returns:
        Llama: The loaded Llama model instance
        str: Path to the downloaded model file
    """
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    
    # Define the model ID and the specific GGUF filename
    model_id = "mradermacher/MedicalQwen3-Reasoning-14B-IT-i1-GGUF"
    model_filename = "MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf"
    
    # Download the GGUF file
    print(f"Downloading {model_filename} from {model_id}...")
    model_path = hf_hub_download(repo_id=model_id, filename=model_filename)
    print(f"Model downloaded to: {model_path}")
    
    # Load the LlamaCpp model with increased context window
    print("Loading LlamaCpp model...")
    llm = Llama(model_path=model_path, n_gpu_layers=0, n_ctx=4096, verbose=False)
    print("Model loaded.")
    
    return llm, model_path

# Load the model once at the beginning
llm, model_path = load_medical_model()

Model downloaded to: C:\Users\kocak\.cache\huggingface\hub\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\snapshots\562d8aa8d3e32cc0945f12a598f43a8c6a259332\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf
Loading LlamaCpp model...


llama_context: n_ctx_per_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Model loaded.


In [9]:
# Function to bracket entities with their labels
def bracket_entities_with_labels(text, entities):
    """
    Adds brackets around entities with their labels.
    
    Args:
        text (str): Original clinical note text
        entities (list): List of NER entities with start, end, and label information
    
    Returns:
        str: Text with entities bracketed with their labels
    """
    # Sort entities by start position in reverse order to avoid index shifting
    sorted_entities = sorted(entities, key=lambda x: x['start'], reverse=True)
    
    bracketed_text = text
    for entity in sorted_entities:
        start = entity['start']
        end = entity['end']
        label = entity['entity_group']
        entity_text = text[start:end]
        
        # Create opening and closing brackets
        opening_bracket = f"[{label}]"
        closing_bracket = f"[/{label}]"
        
        # Insert brackets
        bracketed_text = bracketed_text[:end] + closing_bracket + bracketed_text[end:]
        bracketed_text = bracketed_text[:start] + opening_bracket + bracketed_text[start:]
    
    return bracketed_text

# Apply bracketing to the first clinical note
print("Original entities from NER:")
for ent in entities:
    print(f"  - {ent['word']} ({ent['entity_group']}) at [{ent['start']}:{ent['end']}]")

print("\n" + "="*50)
print("Applying bracketing to clinical note...")
bracketed_note = bracket_entities_with_labels(report1, entities)

print("\nBracketed Clinical Note:")
print(bracketed_note)

Original entities from NER:
  - fallen (THERAPEUTIC_PROCEDURE) at [23:30]
  - occipital region (BIOLOGICAL_STRUCTURE) at [57:74]
  - loss of consciousness (SIGN_SYMPTOM) at [82:104]
  - concussion (DISEASE_DISORDER) at [107:118]
  - right trochanter (BIOLOGICAL_STRUCTURE) at [120:137]
  - right knee (BIOLOGICAL_STRUCTURE) at [140:151]
  - right ankle (BIOLOGICAL_STRUCTURE) at [153:165]
  - CS 15_Cincinnati (DETAILED_DESCRIPTION) at [165:182]
  - negative_fluent speech_ (DETAILED_DESCRIPTION) at [182:206]
  - isochoric (SIGN_SYMPTOM) at [206:215]
  - is (LAB_VALUE) at [216:219]
  - ocyclic (SIGN_SYMPTOM) at [219:226]
  - photoreactive pupils (SIGN_SYMPTOM) at [231:252]
  - rigidity (SIGN_SYMPTOM) at [256:265]
  - Conscious (SIGN_SYMPTOM) at [266:276]
  - lucid (SIGN_SYMPTOM) at [277:283]
  - oriented (SIGN_SYMPTOM) at [284:293]
  - agitated (SIGN_SYMPTOM) at [298:307]
  - rigid (DETAILED_DESCRIPTION) at [316:322]
  - cervical collar (THERAPEUTIC_PROCEDURE) at [322:338]
  - BP (DIAGNOSTI

In [10]:
# Function to clean up model output by removing think statements and formatting issues
def clean_model_output(generated_text):
    """
    Cleans up model output by removing multiple think statements and formatting issues.
    
    Args:
        generated_text (str): Raw text output from the model
    
    Returns:
        str: Cleaned text ready for JSON parsing
    """
    if not generated_text:
        return generated_text
    
    import re
    
    # Remove the specific "think" pattern we observed: "think\n\n"
    cleaned = re.sub(r'think\s*\n\s*\n', '', generated_text, flags=re.IGNORECASE)
    
    # Remove any remaining multiple newlines (more than 2 in a row)
    cleaned = re.sub(r'\n\s*\n\s*\n+', '\n\n', cleaned)
    
    # Remove any leading/trailing whitespace and newlines
    cleaned = cleaned.strip()
    
    # Find the first occurrence of '{' and start from there (in case there's still prefix text)
    first_brace = cleaned.find('{')
    if first_brace != -1:
        cleaned = cleaned[first_brace:]
    
    # Find the last occurrence of '}' and end there (in case there's suffix text)
    last_brace = cleaned.rfind('}')
    if last_brace != -1:
        cleaned = cleaned[:last_brace + 1]
    
    # Final strip
    cleaned = cleaned.strip()
    
    return cleaned

# Test the cleaning function with the actual problematic pattern we observed
test_text = "think\n\nthink\n\n{\n    \"test\": \"value\"\n}"
print("Original text:")
print(repr(test_text))
print("\nCleaned text:")
print(repr(clean_model_output(test_text)))

# Test with the actual pattern from the error logs
actual_pattern = "think\n\n\n{\n    \"chronic metabolic failure\": [\n        {\n            \"triplet\": (\"patient\", \"has_condition\", \"Diabetes mellitus type 2\"),\n            \"evidence\": \"Diabetes mellitus type 2\",\n            \"reasoning\": \"The presence of Diabetes mellitus type 2 in the past medical history indicates a chronic metabolic condition.\"\n        }\n    ]\n}"
print("\nActual pattern test:")
print("Original:")
print(repr(actual_pattern))
print("\nCleaned:")
print(repr(clean_model_output(actual_pattern)))

Original text:
'think\n\nthink\n\n{\n    "test": "value"\n}'

Cleaned text:
'{\n    "test": "value"\n}'

Actual pattern test:
Original:
'think\n\n\n{\n    "chronic metabolic failure": [\n        {\n            "triplet": ("patient", "has_condition", "Diabetes mellitus type 2"),\n            "evidence": "Diabetes mellitus type 2",\n            "reasoning": "The presence of Diabetes mellitus type 2 in the past medical history indicates a chronic metabolic condition."\n        }\n    ]\n}'

Cleaned:
'{\n    "chronic metabolic failure": [\n        {\n            "triplet": ("patient", "has_condition", "Diabetes mellitus type 2"),\n            "evidence": "Diabetes mellitus type 2",\n            "reasoning": "The presence of Diabetes mellitus type 2 in the past medical history indicates a chronic metabolic condition."\n        }\n    ]\n}'


In [11]:
if generated_text:
            try:
                # Clean the generated text to remove think statements and formatting issues
                cleaned_text = clean_model_output(generated_text)
                print(f"\n--- Cleaned Text ---\n'{cleaned_text}'")
                
                # Parse as Python dictionary
                parsed_triplets = eval(cleaned_text)
                if isinstance(parsed_triplets, dict):
                    extracted_triplets = parsed_triplets
                    print("\n--- Parsed Evidence-Backed Relation Triplets ---")
                    for feature, items in extracted_triplets.items():
                        print(f"\nFeature: {feature}")
                        for i, item in enumerate(items, 1):
                            if isinstance(item, dict) and "triplet" in item:
                                print(f"  {i}. {item['triplet']}")
                                print(f"     Evidence: {item.get('evidence', 'N/A')}")
                                print(f"     Reasoning: {item.get('reasoning', 'N/A')}")
                            else:
                                print(f"  {i}. {item}")
                else:
                    print(f"\nOutput is not a dictionary: {type(parsed_triplets)}")
                    extracted_triplets = fallback_evidence_triplet_extraction(generated_text)
            except Exception as e:
                print(f"\nError parsing model output: {e}")
                print("Trying to extract triplets manually...")
                extracted_triplets = fallback_evidence_triplet_extraction(generated_text)

NameError: name 'generated_text' is not defined

In [ ]:
def extract_relation_triplets_with_llm(llm, bracketed_text, ground_truth_annotations):
    """
    Extracts evidence-backed relation triplets from a bracketed clinical note using the Medical Qwen3 LLM.
    
    Args:
        llm: The loaded Llama model instance
        bracketed_text (str): Clinical note with entities bracketed by their labels
        ground_truth_annotations (list): List of annotation dictionaries with 'item' and 'ground_truth' keys
    
    Returns:
        tuple: (extracted_triplets dict, full_model_output dict)
    """
    # System prompt for the medical LLM
    system_prompt = """You are a medical AI assistant specialized in extracting evidence-backed relation triplets from clinical notes. 
Your task is to analyze the provided bracketed clinical note and extract relation triplets that justify the ground truth annotations.

For EACH feature that has a known ground truth (not 'unknown'), you must:
1. Find supporting evidence in the text
2. Create relation triplets (head, relation, tail) 
3. Provide exact quotes as evidence
4. Explain your reasoning

Focus on features that are marked as present ('y', 'n', specific values) in the ground truth annotations."""

    # Prepare ground truth context
    annotated_items = []
    for annotation in ground_truth_annotations:
        if annotation['ground_truth'] != 'unknown':
            annotated_items.append(f"- {annotation['item']}: {annotation['ground_truth']}")
    
    ground_truth_context = "\n".join(annotated_items) if annotated_items else "No specific ground truth annotations available."
    
    # Construct user prompt
    user_prompt = f"""Please extract EVIDENCE-BACKED relation triplets for EACH FEATURE from the following clinical data:

BRACKETED CLINICAL NOTE:
{bracketed_text}

GROUND TRUTH ANNOTATIONS (items with known status):
{ground_truth_context}

CRITICAL: For each feature, explain WHY it was annotated by finding supporting evidence in the text. If a feature like 'chronic metabolic failure' is marked as present but not explicitly mentioned, explain what evidence led to that conclusion.

Return a Python dictionary where each key is a feature name and the value is a list of dictionaries containing:
- triplet: (head, relation, tail)
- evidence: exact quote from text
- reasoning: how this justifies the annotation"""

    # Construct the full chat prompt using Qwen's chat template format
    prompt_template = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"
    
    print("Extracting evidence-backed relation triplets with LLM...")
    print(f"Prompt length: {len(prompt_template)} characters")
    print(f"Ground truth annotations: {len([a for a in ground_truth_annotations if a['ground_truth'] != 'unknown'])} items")
    
    # Generate response
    output = llm(
        prompt_template,
        max_tokens=2048,
        temperature=0.1,
        top_p=0.9,
        repeat_penalty=1.1,
        stop=["<|im_end|>", "\n\n\n"],
        echo=False
    )
    
    print("Full model output:")
    print(output)
    
    # Extract and parse the generated text
    extracted_triplets = {}
    if "choices" in output and len(output["choices"]) > 0:
        generated_text = output["choices"][0]["text"].strip()
        print("\n--- Extracted Relation Triplets (Raw Output) ---")
        print(f"Generated text: '{generated_text}'")
        
        if generated_text:
            try:
                # Clean the generated text to remove think statements and formatting issues
                cleaned_text = clean_model_output(generated_text)
                print(f"\n--- Cleaned Text ---\n'{cleaned_text}'")
                
                # Parse as Python dictionary
                parsed_triplets = eval(cleaned_text)
                if isinstance(parsed_triplets, dict):
                    extracted_triplets = parsed_triplets
                    print("\n--- Parsed Evidence-Backed Relation Triplets ---")
                    for feature, items in extracted_triplets.items():
                        print(f"\nFeature: {feature}")
                        for i, item in enumerate(items, 1):
                            if isinstance(item, dict) and "triplet" in item:
                                print(f"  {i}. {item['triplet']}")
                                print(f"     Evidence: {item.get('evidence', 'N/A')}")
                                print(f"     Reasoning: {item.get('reasoning', 'N/A')}")
                            else:
                                print(f"  {i}. {item}")
                else:
                    print(f"\nOutput is not a dictionary: {type(parsed_triplets)}")
                    extracted_triplets = fallback_evidence_triplet_extraction(generated_text)
            except Exception as e:
                print(f"\nError parsing model output: {e}")
                print("Trying to extract triplets manually...")
                extracted_triplets = fallback_evidence_triplet_extraction(generated_text)
    
    return extracted_triplets, output

def fallback_evidence_triplet_extraction(generated_text):
    """
    Fallback method to extract evidence-backed relation triplets when model output is not valid Python.
    
    Args:
        generated_text (str): The raw text output from the model
    
    Returns:
        dict: Dictionary of extracted triplets
    """
    # This is a simple fallback - in practice you might want more sophisticated parsing
    print("Using fallback extraction method...")
    return {}

In [20]:
# Load the medical LLM model for relation extraction
print("Loading medical LLM for relation extraction...")
medical_llm, _model_path = load_medical_model()

# Extract evidence-backed relation triplets using the bracketed note and ground truth annotations
print("\n" + "="*60)
print("EVIDENCE-BACKED RELATION TRIPLET EXTRACTION")
print("="*60)

ground_truth_annotations = df["annotations"][0]
relation_triplets_by_feature, full_output = extract_relation_triplets_with_llm(
    medical_llm,
    bracketed_note,
    ground_truth_annotations,
)

print("\n" + "="*60)
print("FINAL EVIDENCE-BACKED RELATION TRIPLETS FOR KNOWLEDGE GRAPH")
print("="*60)

if relation_triplets_by_feature:
    total_triplets = sum(len(items) for items in relation_triplets_by_feature.values())
    print(f"Successfully extracted {total_triplets} evidence-backed relation entries across {len(relation_triplets_by_feature)} features:\n")
    for feature, items in relation_triplets_by_feature.items():
        print(f"\nFeature: {feature}")
        for i, item in enumerate(items, 1):
            if isinstance(item, dict) and "triplet" in item:
                head, relation, tail = item["triplet"]
                evidence = item.get("evidence", "N/A")
                reasoning = item.get("reasoning", "N/A")
                print(f"  {i:2d}. ({head}) -[{relation}]-> ({tail})")
                print(f"      Evidence: {evidence}")
                print(f"      Reasoning: {reasoning}")
            else:
                # Fallback for non-dict items
                print(f"  {i:2d}. {item}")
else:
    print("No evidence-backed relation triplets were successfully extracted.")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Original clinical note length: {len(report1)} characters")
print(f"NER entities found: {len(entities)}")
print(f"Bracketed note length: {len(bracketed_note)} characters")
print(f"Ground truth annotations: {len(ground_truth_annotations)} items")
print(f"Annotated items (non-unknown): {len([a for a in ground_truth_annotations if a['ground_truth'] != 'unknown'])}")
print(f"Features with evidence-backed triplets: {len(relation_triplets_by_feature) if relation_triplets_by_feature else 0}")
print(f"Total evidence-backed entries: {sum(len(items) for items in relation_triplets_by_feature.values()) if relation_triplets_by_feature else 0}")

Loading medical LLM for relation extraction...
Model downloaded to: C:\Users\kocak\.cache\huggingface\hub\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\snapshots\562d8aa8d3e32cc0945f12a598f43a8c6a259332\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf
Loading LlamaCpp model...


llama_context: n_ctx_per_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Model loaded.

EVIDENCE-BACKED RELATION TRIPLET EXTRACTION
Extracting evidence-backed relation triplets with LLM...
Prompt length: 5160 characters
Ground truth annotations: 12 items
Full model output:
{'id': 'cmpl-475ae5a7-c3d7-40d9-8764-958563ee9ec1', 'object': 'text_completion', 'created': 1766850329, 'model': 'C:\\Users\\kocak\\.cache\\huggingface\\hub\\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\\snapshots\\562d8aa8d3e32cc0945f12a598f43a8c6a259332\\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf', 'choices': [{'text': '<think>\n\n</think>\n\n{\n    "chronic metabolic failure": [\n        {\n            "triplet": ("Diabetes mellitus type 2", "is a type of", "metabolic disorder"),\n            "evidence": "-[DISEASE_DISORDER] Diabetes mellitus type 2[/DISEASE_DISORDER]",\n            "reasoning": "The presence of diabetes mellitus type 2 in the past medical history indicates chronic metabolic failure."\n        }\n    ],\n    "history of allergy": [\n        {\n          

# Batch Processing for All Reports

Now let's create a function to process all report-annotation pairs in the dataset and extract results to a dictionary.

In [21]:
def process_all_reports_batch(df, ner_pipeline, medical_llm, start_idx=0, end_idx=None, progress_callback=None):
    """
    Process all reports in the dataset using NER+LLM pipeline and extract relations.
    
    Args:
        df: DataFrame containing clinical notes and annotations
        ner_pipeline: NER pipeline for entity extraction
        medical_llm: Medical LLM for relation extraction
        start_idx: Starting index for batch processing
        end_idx: Ending index for batch processing (None for all)
        progress_callback: Optional callback function for progress updates
    
    Returns:
        Dictionary with results for each report
    """
    if end_idx is None:
        end_idx = len(df)
    
    results = {}
    total_reports = end_idx - start_idx
    
    print(f"Starting batch processing of {total_reports} reports (indices {start_idx} to {end_idx-1})")
    
    for i in range(start_idx, end_idx):
        try:
            # Progress update
            if progress_callback:
                progress_callback(i, total_reports)
            elif (i - start_idx + 1) % 5 == 0 or i == end_idx - 1:
                print(f"Processed {i - start_idx + 1}/{total_reports} reports...")
            
            # Get report and annotations
            report_text = df["clinical_note"][i]
            ground_truth_annotations = df["annotations"][i]
            
            # Step 1: Extract entities using NER
            entities = ner_pipeline(report_text)
            
            # Step 2: Create bracketed text
            bracketed_text = bracket_entities_with_labels(report_text, entities)
            
            # Step 3: Extract relation triplets using LLM
            relation_triplets_by_feature, full_output = extract_relation_triplets_with_llm(
                medical_llm,
                bracketed_text,
                ground_truth_annotations,
            )
            
            # Store results
            results[f"report_{i}"] = {
                "report_index": i,
                "original_text": report_text,
                "entities": entities,
                "bracketed_text": bracketed_text,
                "ground_truth_annotations": ground_truth_annotations,
                "relation_triplets_by_feature": relation_triplets_by_feature,
                "full_llm_output": full_output,
                "summary": {
                    "original_text_length": len(report_text),
                    "entities_count": len(entities),
                    "bracketed_text_length": len(bracketed_text),
                    "annotations_count": len(ground_truth_annotations),
                    "annotated_items_count": len([a for a in ground_truth_annotations if a['ground_truth'] != 'unknown']),
                    "features_with_triplets": len(relation_triplets_by_feature) if relation_triplets_by_feature else 0,
                    "total_triplets": sum(len(items) for items in relation_triplets_by_feature.values()) if relation_triplets_by_feature else 0
                }
            }
            
        except Exception as e:
            print(f"Error processing report {i}: {str(e)}")
            results[f"report_{i}"] = {
                "report_index": i,
                "error": str(e),
                "status": "failed"
            }
    
    print(f"Batch processing completed. Processed {total_reports} reports.")
    return results

In [22]:
def aggregate_results_summary(results_dict):
    """
    Aggregate results from batch processing into summary statistics.
    
    Args:
        results_dict: Dictionary containing results from process_all_reports_batch
    
    Returns:
        Dictionary with aggregated statistics and summary
    """
    summary = {
        "total_reports": len(results_dict),
        "successful_reports": 0,
        "failed_reports": 0,
        "total_entities": 0,
        "total_triplets": 0,
        "features_processed": set(),
        "reports_by_feature_count": {},
        "error_reports": []
    }
    
    for report_key, result in results_dict.items():
        if "error" in result:
            summary["failed_reports"] += 1
            summary["error_reports"].append({
                "report_key": report_key,
                "error": result["error"]
            })
        else:
            summary["successful_reports"] += 1
            
            # Aggregate counts
            if "summary" in result:
                summary["total_entities"] += result["summary"]["entities_count"]
                summary["total_triplets"] += result["summary"]["total_triplets"]
                
                # Track features
                feature_count = result["summary"]["features_with_triplets"]
                summary["features_processed"].add(feature_count)
                
                if feature_count not in summary["reports_by_feature_count"]:
                    summary["reports_by_feature_count"][feature_count] = 0
                summary["reports_by_feature_count"][feature_count] += 1
    
    # Convert set to sorted list for better display
    summary["features_processed"] = sorted(list(summary["features_processed"]))
    
    return summary

In [23]:
def save_results_to_file(results_dict, output_file="batch_results.json"):
    """
    Save batch processing results to a JSON file.
    
    Args:
        results_dict: Dictionary containing results from process_all_reports_batch
        output_file: Output file path
    """
    import json
    import os
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else ".", exist_ok=True)
    
    # Save results
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results_dict, f, indent=2, ensure_ascii=False, default=str)
    
    print(f"Results saved to {output_file}")
    
    # Also save summary
    summary = aggregate_results_summary(results_dict)
    summary_file = output_file.replace('.json', '_summary.json')
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False, default=str)
    
    print(f"Summary saved to {summary_file}")
    return output_file, summary_file

In [15]:
from tqdm.auto import tqdm
import time

def progress_with_bar(current, total, start_time=None):
    """Enhanced progress callback with time estimation."""
    if start_time is None:
        start_time = time.time()
    
    elapsed = time.time() - start_time
    progress = (current + 1) / total
    
    if progress > 0:
        eta = elapsed / progress - elapsed
        eta_str = f"ETA: {eta/60:.1f}min" if eta > 60 else f"ETA: {eta:.0f}s"
    else:
        eta_str = "ETA: --"
    
    print(f"Progress: {current + 1}/{total} ({progress*100:.1f}%) - {eta_str}")

def run_batch_processing_with_progress(df, ner_pipeline, medical_llm, 
                                     batch_size=10, start_idx=0, end_idx=None):
    """
    Run batch processing with enhanced progress tracking and error recovery.
    
    Args:
        df: DataFrame containing clinical notes and annotations
        ner_pipeline: NER pipeline for entity extraction
        medical_llm: Medical LLM for relation extraction
        batch_size: Number of reports to process in each batch
        start_idx: Starting index for processing
        end_idx: Ending index for processing (None for all)
    
    Returns:
        Dictionary with all results
    """
    if end_idx is None:
        end_idx = len(df)
    
    all_results = {}
    start_time = time.time()
    
    # Process in batches for better memory management and progress tracking
    for batch_start in range(start_idx, end_idx, batch_size):
        batch_end = min(batch_start + batch_size, end_idx)
        
        print(f"\n{'='*60}")
        print(f"Processing batch: reports {batch_start} to {batch_end-1}")
        print(f"{'='*60}")
        
        # Process current batch
        batch_results = process_all_reports_batch(
            df, ner_pipeline, medical_llm, 
            start_idx=batch_start, 
            end_idx=batch_end,
            progress_callback=lambda current, total: progress_with_bar(current, total, start_time)
        )
        
        # Merge batch results
        all_results.update(batch_results)
        
        # Save intermediate results
        if batch_start > start_idx:
            intermediate_file = f"intermediate_results_batch_{batch_start}_{batch_end-1}.json"
            save_results_to_file(batch_results, intermediate_file)
        
        # Brief pause between batches
        time.sleep(1)
    
    # Final summary
    total_time = time.time() - start_time
    summary = aggregate_results_summary(all_results)
    
    print(f"\n{'='*60}")
    print("BATCH PROCESSING COMPLETE")
    print(f"{'='*60}")
    print(f"Total time: {total_time/60:.1f} minutes")
    print(f"Reports processed: {summary['successful_reports']}/{summary['total_reports']}")
    print(f"Failed reports: {summary['failed_reports']}")
    print(f"Total entities extracted: {summary['total_entities']}")
    print(f"Total relation triplets: {summary['total_triplets']}")
    
    if summary['error_reports']:
        print(f"\nErrors encountered in {len(summary['error_reports'])} reports:")
        for error in summary['error_reports'][:5]:  # Show first 5 errors
            print(f"  - {error['report_key']}: {error['error']}")
        if len(summary['error_reports']) > 5:
            print(f"  ... and {len(summary['error_reports']) - 5} more errors")
    
    return all_results, summary

# Example Usage: Test Batch Processing

Let's test the batch processing system on a small subset of reports first.

In [15]:
# Test batch processing on first 3 reports
print("Testing batch processing system on first 3 reports...")

# Load models (if not already loaded)
try:
    ner_pipeline
    medical_llm
    print("Models already loaded")
except NameError:
    print("Loading models...")
    ner_pipeline = ner_pipeline
    medical_llm, _ = load_medical_model()
    print("Models loaded successfully")

# Run batch processing on small test set
test_results, test_summary = run_batch_processing_with_progress(
    df, 
    ner_pipeline, 
    medical_llm,
    batch_size=3,  # Process 3 reports at a time
    start_idx=0, 
    end_idx=3      # Only process first 3 reports for testing
)

print(f"\nTest completed! Results for {len(test_results)} reports generated.")

Testing batch processing system on first 3 reports...
Loading models...
Model downloaded to: C:\Users\kocak\.cache\huggingface\hub\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\snapshots\562d8aa8d3e32cc0945f12a598f43a8c6a259332\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf
Loading LlamaCpp model...


llama_context: n_ctx_per_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Model loaded.
Models loaded successfully

Processing batch: reports 0 to 2
Starting batch processing of 3 reports (indices 0 to 2)
Progress: 1/3 (33.3%) - ETA: 0s
Extracting evidence-backed relation triplets with LLM...
Prompt length: 7160 characters
Ground truth annotations: 12 items
Full model output:
{'id': 'cmpl-aa3e9a5a-20e0-4c08-8364-fc3e3ce1715c', 'object': 'text_completion', 'created': 1766847849, 'model': 'C:\\Users\\kocak\\.cache\\huggingface\\hub\\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\\snapshots\\562d8aa8d3e32cc0945f12a598f43a8c6a259332\\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf', 'choices': [{'text': '<think>\n\n</think>\n\n{\n    "chronic metabolic failure": [\n        {\n            "triplet": ("patient", "has_condition", "Diabetes mellitus type 2"),\n            "evidence": "Diabetes mellitus type 2",\n            "reasoning": "The presence of Diabetes mellitus type 2 in the past medical history indicates a chronic metabolic condition."\n        }\

In [16]:
# Save test results
test_files = save_results_to_file(test_results, "test_batch_results.json")
print(f"Test results saved to: {test_files}")

# Display summary of test results
print("\n" + "="*60)
print("TEST BATCH PROCESSING SUMMARY")
print("="*60)
print(f"Total reports: {test_summary['total_reports']}")
print(f"Successful: {test_summary['successful_reports']}")
print(f"Failed: {test_summary['failed_reports']}")
print(f"Total entities: {test_summary['total_entities']}")
print(f"Total triplets: {test_summary['total_triplets']}")

# Show details for each successful report
for report_key, result in test_results.items():
    if "error" not in result and "summary" in result:
        print(f"\n{report_key}:")
        print(f"  Entities: {result['summary']['entities_count']}")
        print(f"  Triplets: {result['summary']['total_triplets']}")
        print(f"  Features with triplets: {result['summary']['features_with_triplets']}")

NameError: name 'test_results' is not defined

# Full Dataset Processing

Once you're satisfied with the test results, you can process the entire dataset using the code below. **Warning: This may take considerable time depending on your dataset size and hardware.**

In [25]:

# Process entire dataset
print("Starting full dataset processing...")
print(f"Dataset size: {len(df)} reports")

# Adjust batch_size based on your available memory and processing power
# Smaller batch_size = less memory usage but more overhead
# Larger batch_size = more memory usage but faster processing
full_results, full_summary = run_batch_processing_with_progress(
    df, 
    ner_pipeline, 
    llm,
    batch_size=5,  # Adjust based on your system
    start_idx=0, 
    end_idx=None   # Process all reports
)

# Save final results
final_files = save_results_to_file(full_results, "full_dataset_results.json")
print(f"Full dataset results saved to: {final_files}")

# Display final summary
print("\n" + "="*80)
print("FULL DATASET PROCESSING COMPLETE")
print("="*80)
print(f"Total reports processed: {full_summary['successful_reports']}/{full_summary['total_reports']}")
print(f"Success rate: {full_summary['successful_reports']/full_summary['total_reports']*100:.1f}%")
print(f"Total entities extracted: {full_summary['total_entities']}")
print(f"Total relation triplets: {full_summary['total_triplets']}")
print(f"Average entities per report: {full_summary['total_entities']/full_summary['successful_reports']:.1f}")
print(f"Average triplets per report: {full_summary['total_triplets']/full_summary['successful_reports']:.1f}")

Starting full dataset processing...
Dataset size: 10 reports

Processing batch: reports 0 to 4
Starting batch processing of 5 reports (indices 0 to 4)
Progress: 1/5 (20.0%) - ETA: 0s
Extracting evidence-backed relation triplets with LLM...
Prompt length: 5160 characters
Ground truth annotations: 12 items
Full model output:
{'id': 'cmpl-8951a1b6-da0e-44a3-a10c-99592c04e81e', 'object': 'text_completion', 'created': 1766851547, 'model': 'C:\\Users\\kocak\\.cache\\huggingface\\hub\\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\\snapshots\\562d8aa8d3e32cc0945f12a598f43a8c6a259332\\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf', 'choices': [{'text': '<think>\n\n</think>\n\n{\n    "chronic metabolic failure": [\n        {\n            "triplet": ("Diabetes mellitus type 2", "is a type of", "metabolic disorder"),\n            "evidence": "-[DISEASE_DISORDER] Diabetes mellitus type 2[/DISEASE_DISORDER]",\n            "reasoning": "The presence of diabetes mellitus type 2 in the past 

In [27]:
# Helper function to load and analyze saved results
def load_and_analyze_results(results_file):
    """
    Load previously saved results and provide analysis.
    
    Args:
        results_file: Path to the JSON results file
    
    Returns:
        Dictionary with loaded results and analysis
    """
    import json
    
    with open(results_file, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    summary = aggregate_results_summary(results)
    
    print(f"Loaded results from {results_file}")
    print(f"Total reports: {summary['total_reports']}")
    print(f"Successful: {summary['successful_reports']}")
    print(f"Failed: {summary['failed_reports']}")
    print(f"Total entities: {summary['total_entities']}")
    print(f"Total triplets: {summary['total_triplets']}")
    
    return results, summary

# Example usage:
loaded_results, loaded_summary = load_and_analyze_results("full_dataset_results.json")

Loaded results from full_dataset_results.json
Total reports: 10
Successful: 9
Failed: 1
Total entities: 620
Total triplets: 56
